In [18]:
import torch
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
import torchvision.models as models
from torch import nn

import numpy as np
import torchutils as tu
import matplotlib.pyplot as plt
import os
import random
from PIL import Image
import zipfile
import os

In [19]:
local_zip = '/home/vitaliy/Desktop/elbrus bootcamp/phase_2/projector/phase2week1project/model_intel/archive (1).zip'
output_dir = 'data_intel'
if os.path.exists(local_zip):
    with zipfile.ZipFile(local_zip, 'r') as zip_ref:
        zip_ref.extractall(output_dir)
        print('ok')
        os.remove(local_zip)
else: 
    print('bad')

bad


In [20]:
train_transform = T.Compose([
    T.Resize((150,150)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=15),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std= [0.229, 0.224, 0.225])])

valid_transform = T.Compose([
    T.Resize((150,150)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [21]:
ds = ImageFolder(root='data_intel')

print(len(ds))
print(ds.classes)
print(len(ds.classes))


24335
['seg_pred', 'seg_test', 'seg_train']
3


In [22]:
train_ds = ImageFolder(
    root='data_intel/seg_train/seg_train',
    transform=train_transform
)

valid_ds = ImageFolder(
    root='data_intel/seg_test/seg_test',
    transform=valid_transform
)

print(f'class in ds {train_ds.classes}')
print(f'picture for train {len(train_ds)}')
print(f'picture for validation {len(valid_ds)}')

class in ds ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
picture for train 14034
picture for validation 3000


In [23]:
num_work = os.cpu_count() or 2 

In [24]:
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=num_work, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=256, shuffle=True,pin_memory=True)


In [25]:
device='cuda'

In [26]:
res101 = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
print(res101)

for params in res101.parameters():
    params.requires_grad = False

num_feat = res101.fc.in_features

res101.fc = nn.Linear(num_feat, 6)

for name, param in res101.named_parameters():
    if 'layer4' in name:
        param.requires_grad = True

res101 = res101.to(device)

train_params = [p for p in res101.parameters() if p.requires_grad]

print(len(train_params))



ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

In [27]:
test_batch = next(iter(train_loader))

test_batch_sample, test_batch_label = test_batch

In [28]:
test_batch_sample[0].mean(), test_batch_sample[0].std()

(tensor(-0.0787), tensor(1.0367))

In [29]:
img_batch = test_batch_sample.to(device)

In [30]:
tu.get_model_summary(res101.to(device), img_batch)

Layer                                         Kernel               Output           Params              FLOPs
0_conv1                                      [3, 64, 7, 7]     [256, 64, 75, 75]       9,408   13,547,520,000
1_bn1                                                 [64]     [256, 64, 75, 75]         128      368,640,000
2_relu                                                   -     [256, 64, 75, 75]           0                0
3_maxpool                                                -     [256, 64, 38, 38]           0                0
4_layer1.0.Conv2d_conv1                     [64, 64, 1, 1]     [256, 64, 38, 38]       4,096    1,514,143,744
5_layer1.0.BatchNorm2d_bn1                            [64]     [256, 64, 38, 38]         128       94,633,984
6_layer1.0.ReLU_relu                                     -     [256, 64, 38, 38]           0                0
7_layer1.0.Conv2d_conv2                     [64, 64, 3, 3]     [256, 64, 38, 38]      36,864   13,627,293,696
8_layer1.0

In [31]:
def fit(
        model: torch.nn.Module, 
        n_epochs: int, 
        optimizer: torch.optim.Optimizer,
        train_loader: DataLoader,
        valid_loader: DataLoader
        ) -> tuple[list, ...]:

    train_loss = []
    valid_loss = []
    train_metric = []
    valid_metric = []

    for epoch in range(n_epochs):
        model.train()
        batch_loss = []
        batch_metric = []

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            y_pred = model(images)
            loss = crit(y_pred, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_loss.append(loss.item())
            acc = (y_pred.argmax(dim=1) == labels).float().mean().item()
            batch_metric.append(acc)
        train_loss.append(sum(batch_loss) / len(batch_loss))
        train_metric.append(sum(batch_metric) / len(batch_metric))

        model.eval()
        val_loss = []
        val_metric = []
        with torch.no_grad():
            for images, labels in valid_loader:
                images = images.to(device)
                labels = labels.to(device)

                y_pred = model(images)
                loss = crit(y_pred, labels)

                val_loss.append(loss.item())
                acc = (y_pred.argmax(dim=1) == labels).float().mean().item()
                val_metric.append(acc)
        valid_loss.append(sum(val_loss) / len(val_loss))
        valid_metric.append(sum(val_metric) / len(val_metric))

        print(f'Epoch: {epoch} / {n_epochs}')
        print(f"train_loss {train_loss[-1]:.4f}  train_metric {train_metric[-1]:.4f}")
        print(f"valid_loss {valid_loss[-1]:.4f}  valid_metric {valid_metric[-1]:.4f}")
    
    return train_loss, valid_loss, train_metric, valid_metric
        

In [32]:
optimizer = torch.optim.AdamW(
    train_params,
    lr=1e-5,
    weight_decay=1e-2
)

crit = nn.CrossEntropyLoss()

In [33]:
train_loss, valid_loss, train_metric, valid_metric = fit(
    model = res101,
    n_epochs=10,
    optimizer=optimizer,
    train_loader=train_loader,
    valid_loader=valid_loader
)

Epoch: 0 / 10
train_loss 1.6558  train_metric 0.3642
valid_loss 1.4981  valid_metric 0.5692
Epoch: 1 / 10
train_loss 1.3307  train_metric 0.6691
valid_loss 1.0878  valid_metric 0.7792
Epoch: 2 / 10
train_loss 0.9534  train_metric 0.8036
valid_loss 0.7398  valid_metric 0.8473
Epoch: 3 / 10
train_loss 0.6595  train_metric 0.8496
valid_loss 0.5226  valid_metric 0.8685
Epoch: 4 / 10
train_loss 0.4977  train_metric 0.8724
valid_loss 0.4234  valid_metric 0.8857
Epoch: 5 / 10
train_loss 0.4008  train_metric 0.8851
valid_loss 0.3553  valid_metric 0.8932
Epoch: 6 / 10
train_loss 0.3445  train_metric 0.8948
valid_loss 0.3164  valid_metric 0.8990
Epoch: 7 / 10
train_loss 0.3087  train_metric 0.9019
valid_loss 0.2920  valid_metric 0.9046
Epoch: 8 / 10
train_loss 0.2824  train_metric 0.9059
valid_loss 0.2718  valid_metric 0.9078
Epoch: 9 / 10
train_loss 0.2638  train_metric 0.9119
valid_loss 0.2630  valid_metric 0.9105


In [34]:
torch.save(res101.state_dict(), 'resnet101_intel.pt')
print('ok')

ok
